# Exploration de order_items — phase 1

Exécuter de haut en bas avec le .venv du projet sur C:. Analyse en lecture seule : aucun export CSV ni appel aux validateurs existants.

Grain : une ligne d'article, clé attendue (order_id, order_item_id). Les répétitions de order_id sont normales.

La cellule lit items deux fois (types inférés et texte source) et joint uniquement les colonnes utiles de orders avec validate="many_to_one". Elle échoue si la clé de orders n'est pas unique, afin de ne pas multiplier silencieusement les lignes. Aucun rapprochement avec payments.

Les nombres figurent dans docs/ORDER_ITEMS_EXPLORATION.md. Le code de cette exploration a été exécuté via le Python local pendant l'audit ; ce fichier notebook est fourni sans sorties enregistrées et n'a pas été exécuté via un noyau Jupyter.

Les montants décimaux non entiers sont normaux. shipping_limit_date est une échéance, pas une date d'expédition réelle. Les valeurs extrêmes sont à investiguer, sans suppression automatique.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
items_path = PROJECT_ROOT / "data" / "olist_order_items_dataset.csv"
orders_path = PROJECT_ROOT / "data" / "olist_orders_dataset.csv"
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in [items_path, orders_path]}
items = pd.read_csv(items_path)
orders = pd.read_csv(orders_path)
text = pd.read_csv(items_path, dtype="string", keep_default_na=False)
numeric = ["order_item_id", "price", "freight_value"]
numeric_checks = {}
for col in numeric:
    values = pd.to_numeric(items[col], errors="coerce")
    numeric_checks[col] = {
        "missing": int(items[col].isna().sum()),
        "non_numeric_non_null": int((items[col].notna() & values.isna()).sum()),
        "non_finite_non_null": int((values.notna() & ~np.isfinite(values)).sum()),
        "negative": int((values < 0).sum()),
        "zero": int(values.eq(0).sum()),
        "non_integer": int((values.notna() & values.mod(1).ne(0)).sum()),
    }
shipping = pd.to_datetime(items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce", exact=True)
date_checks = {
    "missing": int(items.shipping_limit_date.isna().sum()),
    "pattern_mismatch": int((~text.shipping_limit_date.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", na=False)).sum()),
    "invalid_non_null": int((items.shipping_limit_date.notna() & shipping.isna()).sum()),
    "min": str(shipping.min()), "max": str(shipping.max()), "timezone": str(shipping.dt.tz),
}
groups = items.groupby("order_id")["order_item_id"].agg(["size", "min", "max", "nunique"])
sequence_bad = groups[(groups["min"] != 1) | (groups["max"] != groups["size"]) | (groups["nunique"] != groups["size"])]
joined = items.assign(_shipping=shipping).merge(
    orders[["order_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date"]],
    on="order_id", how="left", validate="many_to_one", indicator=True,
)
purchase = pd.to_datetime(joined.order_purchase_timestamp, errors="coerce")
approval = pd.to_datetime(joined.order_approved_at, errors="coerce")
carrier = pd.to_datetime(joined.order_delivered_carrier_date, errors="coerce")
before_purchase = joined._shipping < purchase
before_approval = joined._shipping < approval
after_limit = carrier > joined._shipping
absent = orders.loc[~orders.order_id.isin(items.order_id)]
counts = items.groupby("order_id").size()
report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "executable": sys.executable},
    "source_sha256": source_hashes,
    "source_size_bytes": items_path.stat().st_size,
    "memory_deep_MiB": round(items.memory_usage(deep=True).sum() / 2**20, 3),
    "shape": list(items.shape),
    "dtypes": {c: str(t) for c,t in items.dtypes.items()},
    "missing": {c: int(n) for c,n in items.isna().sum().items()},
    "blank_or_whitespace": {c: int(text[c].str.strip().eq("").sum()) for c in text.columns},
    "id_format_mismatch": {c: int((~text[c].str.fullmatch(r"[0-9a-f]{32}", na=False)).sum()) for c in ["order_id", "product_id", "seller_id"]},
    "exact_duplicate_rows_excess": int(items.duplicated().sum()),
    "duplicate_composite_key_rows": int(items.duplicated(["order_id","order_item_id"], keep=False).sum()),
    "repeated_order_id_rows_excess": int(items.order_id.duplicated().sum()),
    "unique_orders": int(items.order_id.nunique()),
    "unique_products": int(items.product_id.nunique()),
    "unique_sellers": int(items.seller_id.nunique()),
    "orders_with_multiple_items": int(counts.gt(1).sum()),
    "max_items_per_order": int(counts.max()),
    "bad_sequence_orders": int(len(sequence_bad)),
    "numeric_statistics": items[numeric].describe(percentiles=[.25,.5,.75,.95,.99]).to_dict(),
    "numeric_checks": numeric_checks,
    "money_more_than_2_decimals": {c: int((~text[c].str.fullmatch(r"\d+(?:\.\d{1,2})?", na=False)).sum()) for c in ["price","freight_value"]},
    "dates": date_checks,
    "relations": {
        "orders_rows": int(len(orders)), "orders_duplicate_id_rows": int(orders.order_id.duplicated(keep=False).sum()),
        "joined_rows": int(len(joined)),
        "unknown_order_rows": int(joined["_merge"].eq("left_only").sum()),
        "orders_without_items": int(len(absent)),
        "orders_without_items_by_status": {str(s): int(n) for s,n in absent.order_status.value_counts(dropna=False).items()},
        "shipping_limit_before_purchase_rows": int(before_purchase.sum()),
        "shipping_limit_before_purchase_orders": int(joined.loc[before_purchase, "order_id"].nunique()),
        "shipping_limit_before_purchase_by_status": {str(s): int(n) for s,n in joined.loc[before_purchase,"order_status"].value_counts().items()},
        "shipping_limit_before_approval_rows": int(before_approval.sum()),
        "carrier_after_shipping_limit_rows": int(after_limit.sum()),
        "carrier_after_shipping_limit_orders": int(joined.loc[after_limit,"order_id"].nunique()),
    },
}
assert source_hashes == {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in [items_path, orders_path]}, "Source changed"
report["dates"]["by_year"] = {str(y): int(n) for y,n in shipping.dt.year.value_counts().sort_index().items()}
gap_days = (joined._shipping - purchase).dt.total_seconds() / 86400
report["dates"]["purchase_gap_days"] = gap_days.describe(percentiles=[.5,.95,.99]).to_dict()
report["relations"]["shipping_limit_before_approval_orders"] = int(joined.loc[before_approval,"order_id"].nunique())
print(json.dumps(report, indent=2, ensure_ascii=False))



## Avant le futur validateur
Définir les contrôles de clé, références, nombres finis, montants et date obligatoire ; puis décider séparément des alertes chronologiques et des valeurs extrêmes. Les 775 commandes sans articles relèvent d'un rapport de complétude côté orders.
Aucune borne 21, aucun seuil de prix et aucune année limite ne deviennent automatiquement une règle métier.
